# Notebook 03 – Weighted Overlay Analysis (Per 5-Year Period)

Computes a weighted overlay for each 5-year period (1990–2020) using AHP weights from `data/interim/AHP_Vægtninger.xlsx` and reclassified rasters from `data/processed/tsc_reclass/`.

Outputs per period saved to `results/metrics/overlay_output/`:
- `weighted_overlay_{period}.tif`
- `weighted_overlay_clusters_{period}.gpkg` / `.geojson`
- `weighted_overlay_statistics_{period}.csv`
- `weighted_overlay_map_{period}.html`

## Imports & paths

In [8]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import fiona
import folium
import openpyxl
from pathlib import Path
from rasterstats import zonal_stats
from shapely.geometry import shape as geom_from_shape

REPO_ROOT     = Path('..').resolve()
DATA_DIR      = REPO_ROOT / 'data'
RAW_DIR       = DATA_DIR / 'raw'
RESULTS_DIR   = REPO_ROOT / 'results'

RECLASS_FOLDER = DATA_DIR / 'processed' / 'tsc_reclass'
OUTPUT_FOLDER  = RESULTS_DIR / 'metrics' / 'overlay_output'
CLUSTERS_SHP   = RAW_DIR / 'nabolag_inkl_y_kom_shapefile' / 'cluster_outer_v1.shp'
WEIGHTS_EXCEL  = DATA_DIR / 'interim' / 'AHP_Vægtninger.xlsx'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f'Repo root:      {REPO_ROOT}')
print(f'Reclass folder: {RECLASS_FOLDER}')
print(f'Output folder:  {OUTPUT_FOLDER}')
print(f'Clusters SHP:   {CLUSTERS_SHP.exists()}')
print(f'Weights Excel:  {WEIGHTS_EXCEL.exists()}')

Repo root:      C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model
Reclass folder: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_reclass
Output folder:  C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\results\metrics\overlay_output
Clusters SHP:   True
Weights Excel:  True


## Load AHP weights from Excel

In [10]:
wb = openpyxl.load_workbook(WEIGHTS_EXCEL)
ws = wb.active

weight_data = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0] is not None:
        weight_data.append((str(row[0]), float(row[1])))

weights_dict_raw = {var: weight for var, weight in weight_data}

print(f'Weights loaded: {len(weights_dict_raw)} variables')
for var, w in sorted(weights_dict_raw.items(), key=lambda x: x[1], reverse=True):
    print(f'  {var:30s}: {w:.4f}')

Weights loaded: 20 variables
  disp_inc                      : 12.0500
  public_housing                : 9.7167
  mean_sqm                      : 8.8833
  mean_price                    : 8.7833
  mig_out                       : 7.1167
  unemp                         : 6.7833
  lvu                           : 6.3833
  emp                           : 6.1667
  mig_in                        : 5.8000
  ool                           : 4.9500
  age_26_40                     : 3.7333
  grund                         : 3.4667
  gym_erhv                      : 2.7000
  crime_main_y                  : 2.3500
  age_41_55                     : 2.2333
  PUB                           : 2.2000
  PMB                           : 2.0667
  age_18_25                     : 1.7333
  age_56_69                     : 1.5000
  EMUB                          : 1.4167


## Exclude layers & normalize weights

Excluded: `qol`, `mig_net`, `counts` (same as OLD_03)

In [11]:
excluded = ['qol', 'mig_net', 'counts']

weight_dict = {k: v for k, v in weights_dict_raw.items() if k not in excluded}

total_weight = sum(weight_dict.values())
normalized_weights = {k: v / total_weight for k, v in weight_dict.items()}

print(f'Variables after exclusion: {len(weight_dict)} (excluded: {excluded})')
print(f'Sum of normalized weights: {sum(normalized_weights.values()):.6f}')
print('\nNormalized weights (sorted):')
for var, w in sorted(normalized_weights.items(), key=lambda x: x[1], reverse=True):
    print(f'  {var:30s}: {w:.4f}')

Variables after exclusion: 20 (excluded: ['qol', 'mig_net', 'counts'])
Sum of normalized weights: 1.000000

Normalized weights (sorted):
  disp_inc                      : 0.1205
  public_housing                : 0.0971
  mean_sqm                      : 0.0888
  mean_price                    : 0.0878
  mig_out                       : 0.0711
  unemp                         : 0.0678
  lvu                           : 0.0638
  emp                           : 0.0616
  mig_in                        : 0.0580
  ool                           : 0.0495
  age_26_40                     : 0.0373
  grund                         : 0.0347
  gym_erhv                      : 0.0270
  crime_main_y                  : 0.0235
  age_41_55                     : 0.0223
  PUB                           : 0.0220
  PMB                           : 0.0207
  age_18_25                     : 0.0173
  age_56_69                     : 0.0150
  EMUB                          : 0.0142


## Discover rasters grouped by period

Filename pattern: `reclass_{var}_{start}_{end}_tsc.tif`

In [12]:
PERIODS = ['1990_1995', '1995_2000', '2000_2005', '2005_2010', '2010_2015', '2015_2020']

rasters_by_period = {p: {} for p in PERIODS}

for fname in os.listdir(RECLASS_FOLDER):
    if not fname.endswith('.tif'):
        continue
    for period in PERIODS:
        suffix = f'_{period}_tsc.tif'
        if fname.endswith(suffix):
            var = fname[len('reclass_'):-len(suffix)]
            rasters_by_period[period][var] = RECLASS_FOLDER / fname
            break

print('Rasters discovered per period:')
for period, rasters in rasters_by_period.items():
    start, end = period.split('_')
    print(f'  {start}–{end}: {len(rasters)} rasters')

Rasters discovered per period:
  1990–1995: 21 rasters
  1995–2000: 21 rasters
  2000–2005: 21 rasters
  2005–2010: 21 rasters
  2010–2015: 21 rasters
  2015–2020: 21 rasters


## Load clusters shapefile

In [13]:
geometries = []
properties_list = []

with fiona.open(CLUSTERS_SHP) as src:
    shp_crs = src.crs
    for feature in src:
        geometries.append(geom_from_shape(feature['geometry']))
        properties_list.append(feature['properties'])

clusters_gdf = gpd.GeoDataFrame(properties_list, geometry=geometries, crs=shp_crs)

print(f'Clusters loaded: {len(clusters_gdf)} features')
print(f'CRS: {clusters_gdf.crs}')
print(f'Columns: {list(clusters_gdf.columns)}')


def get_color(value):
    """5-tier red→green color ramp based on normalized overlay value."""
    if pd.isna(value) or value < 0.2:
        return '#d73027'
    elif value < 0.4:
        return '#fc8d59'
    elif value < 0.6:
        return '#fee090'
    elif value < 0.8:
        return '#91bfdb'
    else:
        return '#1a9850'

Clusters loaded: 2232 features
CRS: PROJCS["ETRS89 / UTM zone 32N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25832"]]
Columns: ['area', 'cluster_id', 'fid', 'id_munic', 'munic_code', 'geometry']


## Per-period weighted overlay loop

In [15]:
summary_rows = []

for period in PERIODS:
    start, end = period.split('_')
    print(f"\n{'='*60}")
    print(f'Processing period: {start}–{end}')
    print(f"{'='*60}")

    period_rasters = rasters_by_period.get(period, {})
    if not period_rasters:
        print(f'  No rasters found for {period}, skipping.')
        continue

    # Match normalized weights to rasters available for this period; re-normalize
    matched = {k: normalized_weights[k] for k in normalized_weights if k in period_rasters}
    if not matched:
        print(f'  No weight matches for {period}, skipping.')
        continue

    total_w = sum(matched.values())
    matched_norm = {k: v / total_w for k, v in matched.items()}

    print(f'  Rasters found:   {len(period_rasters)}')
    print(f'  Weights matched: {len(matched_norm)}')

    # --- Grid properties from first raster ---
    first_path = list(period_rasters.values())[0]
    with rasterio.open(first_path) as src:
        raster_crs = src.crs
        raster_transform = src.transform
        height = src.height
        width = src.width

    # --- Weighted overlay (per-pixel weight normalisation) ---
    # Accumulate weighted sum AND the sum of weights that actually had valid data
    # per pixel. Dividing at the end ensures boundary pixels with missing variables
    # are on the same scale as fully-covered interior pixels, eliminating spurious
    # within-cluster variation caused by edge nodata in individual rasters.
    overlay    = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    for var, weight in sorted(matched_norm.items(), key=lambda x: x[1], reverse=True):
        with rasterio.open(period_rasters[var]) as src:
            data = src.read(1).astype(np.float32)
        valid = data > 0
        overlay    += np.where(valid, data * weight, 0.0)
        weight_sum += np.where(valid, weight,        0.0)

    # Normalise: weighted mean using only contributing variables per pixel
    overlay = np.where(weight_sum > 0, overlay / weight_sum, np.nan)

    print(f'  Overlay range: {np.nanmin(overlay):.4f} – {np.nanmax(overlay):.4f}')
    print(f'  Overlay mean:  {np.nanmean(overlay):.4f}')

    # --- Save overlay raster (NaN → 0) ---
    overlay_save = np.where(np.isnan(overlay), 0, overlay).astype(np.float32)
    overlay_path = OUTPUT_FOLDER / f'weighted_overlay_{period}.tif'

    with rasterio.open(
        overlay_path, 'w',
        driver='GTiff', height=height, width=width,
        count=1, dtype=np.float32,
        crs=raster_crs, transform=raster_transform
    ) as dst:
        dst.write(overlay_save, 1)
    print(f'  Saved raster: {overlay_path.name}')

    # --- Zonal statistics ---
    period_gdf = clusters_gdf.copy()
    if period_gdf.crs != raster_crs:
        period_gdf = period_gdf.to_crs(raster_crs)

    stats_list = zonal_stats(
        period_gdf.geometry,
        str(overlay_path),
        affine=raster_transform,
        stats=['mean', 'count', 'std', 'min', 'max'],
        nodata=0,
        all_touched=False
    )
    stats_df = pd.DataFrame(stats_list)
    stats_df['range'] = stats_df['max'] - stats_df['min']

    result_gdf = period_gdf.copy()
    for col in stats_df.columns:
        result_gdf[col] = stats_df[col].values

    # --- Save vector outputs ---
    gpkg_path    = OUTPUT_FOLDER / f'weighted_overlay_clusters_{period}.gpkg'
    geojson_path = OUTPUT_FOLDER / f'weighted_overlay_clusters_{period}.geojson'
    csv_path     = OUTPUT_FOLDER / f'weighted_overlay_statistics_{period}.csv'

    # Drop 'fid' column if present — conflicts with GeoPackage/GeoJSON auto-FID
    fid_cols = [c for c in result_gdf.columns if c.lower() == 'fid']
    save_gdf = result_gdf.drop(columns=fid_cols) if fid_cols else result_gdf

    save_gdf.to_file(str(gpkg_path), driver='GPKG')
    save_gdf.to_file(str(geojson_path), driver='GeoJSON')
    save_gdf.drop(columns='geometry').to_csv(str(csv_path), index=False)
    print(f'  Saved: {gpkg_path.name}, {geojson_path.name}, {csv_path.name}')

    # --- Folium map ---
    map_gdf = result_gdf.to_crs('EPSG:4326')
    center_lat = map_gdf.geometry.centroid.y.mean()
    center_lon = map_gdf.geometry.centroid.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')

    min_val = float(map_gdf['mean'].min())
    max_val = float(map_gdf['mean'].max())
    denom = (max_val - min_val) if max_val > min_val else 1.0
    map_gdf = map_gdf.copy()
    map_gdf['normalized_mean'] = (map_gdf['mean'] - min_val) / denom

    for idx, row in map_gdf.iterrows():
        color = get_color(row['normalized_mean'])
        count_val = int(row['count']) if not pd.isna(row['count']) else 'N/A'
        popup_text = (
            f"<b>Period: {start}–{end}</b><br><hr>"
            f"Mean: {row['mean']:.4f}<br>"
            f"Std: {row['std']:.4f}<br>"
            f"Count: {count_val}<br>"
            f"Range: {row['range']:.4f}<br>"
            f"Min: {row['min']:.4f} &nbsp; Max: {row['max']:.4f}"
        )
        folium.GeoJson(
            data=row.geometry.__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c, 'color': 'black',
                'weight': 1.5, 'opacity': 0.9, 'fillOpacity': 0.7
            },
            popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)

    legend_html = f"""
    <div style="position:fixed;bottom:50px;right:50px;width:230px;
                background:white;border:2px solid grey;z-index:9999;
                font-size:13px;padding:10px;border-radius:5px;">
      <p style="margin:0;font-weight:bold;text-align:center;">Period {start}–{end}<br>Mean Overlay Value</p>
      <hr style="margin:5px 0;">
      <p style="margin:3px 0;"><i style="background:#d73027;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very Low (0–20%)</p>
      <p style="margin:3px 0;"><i style="background:#fc8d59;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Low (20–40%)</p>
      <p style="margin:3px 0;"><i style="background:#fee090;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Medium (40–60%)</p>
      <p style="margin:3px 0;"><i style="background:#91bfdb;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>High (60–80%)</p>
      <p style="margin:3px 0;"><i style="background:#1a9850;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very High (80–100%)</p>
      <hr style="margin:8px 0;">
      <p style="margin:3px 0;font-size:11px;"><b>Min:</b> {min_val:.4f}</p>
      <p style="margin:3px 0;font-size:11px;"><b>Max:</b> {max_val:.4f}</p>
      <p style="margin:3px 0;font-size:11px;"><b>Clusters:</b> {len(map_gdf)}</p>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    map_path = OUTPUT_FOLDER / f'weighted_overlay_map_{period}.html'
    m.save(str(map_path))
    print(f'  Saved map: {map_path.name}')

    std_valid  = stats_df['std'].dropna()
    std_nonzero = int((std_valid > 0).sum())
    total_clusters = int(std_valid.count())

    summary_rows.append({
        'period': f'{start}–{end}',
        'rasters_matched': len(matched_norm),
        'clusters_with_data': int(stats_df['count'].notna().sum()),
        'mean_overlay': round(float(np.nanmean(overlay_save)), 4),
        'max_overlay': round(float(np.nanmax(overlay_save)), 4),
        'std_nonzero': std_nonzero,
        'total_clusters': total_clusters,
    })

print(f"\n{'='*60}")
print('All periods processed.')



Processing period: 1990–1995
  Rasters found:   21
  Weights matched: 20
  Overlay range: 2.0000 – 4.6886
  Overlay mean:  3.3616
  Saved raster: weighted_overlay_1990_1995.tif


C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:50: RuntimeWarning: invalid value encountered in divide
  overlay = np.where(weight_sum > 0, overlay / weight_sum, np.nan)


  Saved: weighted_overlay_clusters_1990_1995.gpkg, weighted_overlay_clusters_1990_1995.geojson, weighted_overlay_statistics_1990_1995.csv


C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:104: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = map_gdf.geometry.centroid.y.mean()
C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:105: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = map_gdf.geometry.centroid.x.mean()


  Saved map: weighted_overlay_map_1990_1995.html

Processing period: 1995–2000
  Rasters found:   21
  Weights matched: 20
  Overlay range: 1.0000 – 5.0713
  Overlay mean:  3.1425
  Saved raster: weighted_overlay_1995_2000.tif


...

  Saved map: weighted_overlay_map_2010_2015.html

Processing period: 2015–2020
  Rasters found:   21
  Weights matched: 20
  Overlay range: 1.0000 – 4.8112
  Overlay mean:  3.1172
  Saved raster: weighted_overlay_2015_2020.tif


C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:50: RuntimeWarning: invalid value encountered in divide
  overlay = np.where(weight_sum > 0, overlay / weight_sum, np.nan)


  Saved: weighted_overlay_clusters_2015_2020.gpkg, weighted_overlay_clusters_2015_2020.geojson, weighted_overlay_statistics_2015_2020.csv


C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:104: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = map_gdf.geometry.centroid.y.mean()
C:\Users\jonas\AppData\Local\Temp\ipykernel_25384\294062316.py:105: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = map_gdf.geometry.centroid.x.mean()


  Saved map: weighted_overlay_map_2015_2020.html

All periods processed.


## Summary

In [16]:
summary_df = pd.DataFrame(summary_rows)
print('Weighted Overlay Summary')
print('=' * 60)
print(summary_df.to_string(index=False))

print('\nClusters with std > 0 per period:')
for _, row in summary_df.iterrows():
    print(f"  {row['period']}: {row['std_nonzero']}/{row['total_clusters']} std not zero")

outputs = sorted(OUTPUT_FOLDER.glob('weighted_overlay_*.tif'))
print(f'\nOutput rasters in {OUTPUT_FOLDER.relative_to(REPO_ROOT)}:')
for f in outputs:
    print(f'  {f.name}')

Weighted Overlay Summary
   period  rasters_matched  clusters_with_data  mean_overlay  max_overlay  std_nonzero  total_clusters
1990–1995               20                2232        1.1102       4.6886          897            2232
1995–2000               20                2232        1.0378       5.0713          898            2232
2000–2005               20                2232        1.0679       5.3194          879            2232
2005–2010               20                2232        1.1696       4.9923          904            2232
2010–2015               20                2232        1.1305       6.0000          922            2232
2015–2020               20                2232        1.0295       4.8112          899            2232

Clusters with std > 0 per period:
  1990–1995: 897/2232 std not zero
  1995–2000: 898/2232 std not zero
  2000–2005: 879/2232 std not zero
  2005–2010: 904/2232 std not zero
  2010–2015: 922/2232 std not zero
  2015–2020: 899/2232 std not zero

Output r